In [1]:
# # This Python 3 environment comes with many helpful analytics libraries installed
# # It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# # For example, here's several helpful packages to load

# import numpy as np # linear algebra
# import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# # Input data files are available in the read-only "../input/" directory
# # For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

# import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

# # You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# # You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# # Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# # Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

# import kagglehub
# # kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
import os
import json
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.model_selection import train_test_split
from sklearn.utils import class_weight  
import pathlib
import concurrent.futures

# Tắt tạm các cảnh báo lỗi rác của C++ dưới nền để màn hình không bị trôi
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

# ==========================================
# 1. CẤU HÌNH ĐƯỜNG DẪN, THAM SỐ & MULTI-GPU
# ==========================================
DATASET_ROOT = "/kaggle/input/datasets/nguynnhtpht/dataset-for-nutridish-project/Model_Training/datasets"
VN_FOODS_DIR = os.path.join(DATASET_ROOT, "30vnfoods/Images") 
FOOD101_DIR = os.path.join(DATASET_ROOT, "food101/images")

# Tham số huấn luyện cơ bản
IMG_SIZE = (224, 224)
EPOCHS = 20  
LEARNING_RATE = 1e-4

print("--- Khởi tạo cấu hình và kiểm tra GPU ---")
print("Số lượng GPU vật lý khả dụng:", len(tf.config.list_physical_devices('GPU')))

# KÍCH HOẠT MULTI-GPU
strategy = tf.distribute.MirroredStrategy()
print(f"Số lượng GPU đang được đồng bộ hóa (Replicas): {strategy.num_replicas_in_sync}")

# Cấu hình BATCH_SIZE chuẩn cho Multi-GPU
BATCH_SIZE_PER_REPLICA = 32
BATCH_SIZE = BATCH_SIZE_PER_REPLICA * strategy.num_replicas_in_sync
print(f"Tổng Batch Size thực tế: {BATCH_SIZE}")

# ==========================================
# 2. ĐỒNG NHẤT DANH SÁCH NHÃN (131 CLASSES)
# ==========================================
def build_global_class_mapping(vn_dir, f101_dir):
    vn_train_dir = os.path.join(vn_dir, "Train")
    vn_classes = sorted([d for d in os.listdir(vn_train_dir) if os.path.isdir(os.path.join(vn_train_dir, d))])
    
    f101_classes = sorted([d for d in os.listdir(f101_dir) if os.path.isdir(os.path.join(f101_dir, d))])
    
    global_classes = sorted(list(set(vn_classes + f101_classes)))
    class_to_idx = {class_name: idx for idx, class_name in enumerate(global_classes)}
    
    print(f"Tổng số món ăn Việt Nam: {len(vn_classes)}")
    print(f"Tổng số món ăn quốc tế (Food-101): {len(f101_classes)}")
    print(f"Tổng số lượng nhãn sau khi gộp: {len(global_classes)}")
    
    return global_classes, class_to_idx

GLOBAL_CLASSES, CLASS_TO_IDX = build_global_class_mapping(VN_FOODS_DIR, FOOD101_DIR)

with open("/kaggle/working/labels.txt", "w", encoding="utf-8") as f:
    for class_name in GLOBAL_CLASSES:
        f.write(f"{class_name}\n")
print("Đã lưu cấu trúc nhãn vào /kaggle/working/labels.txt")


# ==========================================
# 3. THU THẬP ĐƯỜNG DẪN ẢNH VÀ GẮN NHÃN ĐỒNG NHẤT
# ==========================================
def collect_f101_image_paths(data_dir, class_to_idx):
    image_paths = []
    labels = []
    for class_name in os.listdir(data_dir):
        class_path = os.path.join(data_dir, class_name)
        if os.path.isdir(class_path):
            target_label = class_to_idx[class_name]
            for img_name in os.listdir(class_path):
                if img_name.lower().endswith(('.png', '.jpg', '.jpeg', '.webp')):
                    image_paths.append(os.path.join(class_path, img_name))
                    labels.append(target_label)
    return image_paths, labels

def collect_vn_image_paths(data_dir, class_to_idx):
    image_paths = []
    labels = []
    for split_folder in ["Train", "Test", "Validate"]:
        split_dir = os.path.join(data_dir, split_folder)
        if not os.path.exists(split_dir): 
            continue
            
        for class_name in os.listdir(split_dir):
            class_path = os.path.join(split_dir, class_name)
            if os.path.isdir(class_path) and class_name in class_to_idx:
                target_label = class_to_idx[class_name]
                for img_name in os.listdir(class_path):
                    if img_name.lower().endswith(('.png', '.jpg', '.jpeg', '.webp')):
                        image_paths.append(os.path.join(class_path, img_name))
                        labels.append(target_label)
    return image_paths, labels

print("\n--- Đang quét và gom toàn bộ file ảnh ---")
vn_paths, vn_labels = collect_vn_image_paths(VN_FOODS_DIR, CLASS_TO_IDX)
f101_paths, f101_labels = collect_f101_image_paths(FOOD101_DIR, CLASS_TO_IDX)

all_paths = vn_paths + f101_paths
all_labels = vn_labels + f101_labels

# ==========================================
# 3.5 THANH TRỪNG ẢNH LỖI (MỨC ĐỘ CỰC ĐOAN VỚI TENSORFLOW)
# ==========================================
print(f"\n--- Tìm thấy {len(all_paths)} ảnh. Ép TensorFlow soi từng ảnh một... ---")
print("(Quá trình này tốn khoảng 3-5 phút, hãy kiên nhẫn đi pha tách trà nhé!)")

def check_image_tf(data):
    path, label = data
    try:
        img_raw = tf.io.read_file(path)
        _ = tf.io.decode_image(img_raw, channels=3, expand_animations=False)
        return path, label
    except Exception:
        return None

all_data_raw = list(zip(all_paths, all_labels))
clean_paths = []
clean_labels = []

with concurrent.futures.ThreadPoolExecutor(max_workers=64) as executor:
    for result in executor.map(check_image_tf, all_data_raw):
        if result is not None:
            clean_paths.append(result[0])
            clean_labels.append(result[1])

print(f"Tổng số ảnh rác/lỗi đã bị tiêu diệt: {len(all_paths) - len(clean_paths)}")
print(f"Tổng số ảnh SẠCH đưa vào huấn luyện: {len(clean_paths)}")

all_paths = clean_paths
all_labels = clean_labels

# CHIA TẬP DATA SAU KHI ĐÃ LÀM SẠCH
train_paths, val_paths, train_labels, val_labels = train_test_split(
    all_paths, all_labels, test_size=0.2, random_state=42, stratify=all_labels
)


# ==========================================
# 4. XÂY DỰNG TF.DATA PIPELINE HIỆU NĂNG CAO
# ==========================================
def load_and_preprocess_image(path, label):
    image = tf.io.read_file(path)
    image = tf.io.decode_image(image, channels=3, expand_animations=False)
    image.set_shape([None, None, 3])
    image = tf.image.resize(image, IMG_SIZE)
    image = tf.keras.applications.mobilenet_v2.preprocess_input(image)
    return image, label

train_ds = tf.data.Dataset.from_tensor_slices((train_paths, train_labels))
val_ds = tf.data.Dataset.from_tensor_slices((val_paths, val_labels))

AUTOTUNE = tf.data.AUTOTUNE

data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.15),
    layers.RandomZoom(0.1),
])

train_ds = train_ds.shuffle(buffer_size=1000)\
                   .map(load_and_preprocess_image, num_parallel_calls=AUTOTUNE)\
                   .ignore_errors()\
                   .batch(BATCH_SIZE)\
                   .map(lambda x, y: (data_augmentation(x, training=True), y), num_parallel_calls=AUTOTUNE)\
                   .prefetch(buffer_size=AUTOTUNE)

val_ds = val_ds.map(load_and_preprocess_image, num_parallel_calls=AUTOTUNE)\
               .ignore_errors()\
               .batch(BATCH_SIZE)\
               .prefetch(buffer_size=AUTOTUNE)

print("\n--- Đã xây dựng xong luồng nạp dữ liệu tối ưu ---")


# ==========================================
# 5. KHỞI TẠO MÔ HÌNH (TRANSFER LEARNING TRÊN MULTI-GPU)
# ==========================================
print("\n--- Đang khởi tạo cấu trúc mạng MobileNetV2 trên TẤT CẢ GPU ---")

with strategy.scope():
    base_model = tf.keras.applications.MobileNetV2(
        input_shape=(224, 224, 3),
        include_top=False,
        weights='imagenet'
    )

    base_model.trainable = True

    model = models.Sequential([
        base_model,
        layers.GlobalAveragePooling2D(),
        layers.Dropout(0.4), 
        layers.Dense(len(GLOBAL_CLASSES), activation='softmax')
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE), 
        loss=tf.keras.losses.SparseCategoricalCrossentropy(),
        metrics=['accuracy']
    )

model.summary()


# ==========================================
# 6. TÍNH TRỌNG SỐ VÀ HUẤN LUYỆN MÔ HÌNH
# ==========================================
print("\n--- Tính toán Class Weights cho tập dữ liệu mất cân bằng ---")
unique_labels = np.unique(train_labels)
class_weights = class_weight.compute_class_weight(
    class_weight='balanced',
    classes=unique_labels,
    y=train_labels
)
class_weights_dict = dict(zip(unique_labels, class_weights))
print("Hoàn tất tính toán Class Weights!")

print("\n--- Bắt đầu quá trình huấn luyện mạng ---")

callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=2, min_lr=1e-6)
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks, 
    class_weight=class_weights_dict
)


# ==========================================
# 7. CHUYỂN ĐỔI SANG ĐỊNH DẠNG TENSORFLOW LITE (.TFLITE)
# ==========================================
print("\n--- Đang tiến hành nén và chuyển đổi sang file .tflite cho Android ---")

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.target_spec.supported_types = [tf.float16]

tflite_model = converter.convert()

tflite_output_path = "/kaggle/working/food_model.tflite"
with open(tflite_output_path, "wb") as f:
    f.write(tflite_model)

print(f"Chuyển đổi thành công! Mô hình TFLite đã được lưu tại: {tflite_output_path}")
print("Bạn có thể vào tab 'Output' (phía bên phải giao diện Kaggle) để tải 2 file 'food_model.tflite' và 'labels.txt' về máy.")

2026-06-24 13:47:42.419134: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1782308862.583126      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1782308862.633650      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1782308863.038213      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1782308863.038247      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1782308863.038251      23 computation_placer.cc:177] computation placer alr

--- Khởi tạo cấu hình và kiểm tra GPU ---
Số lượng GPU vật lý khả dụng: 2
INFO:tensorflow:Using MirroredStrategy with devices ('/job:localhost/replica:0/task:0/device:GPU:0', '/job:localhost/replica:0/task:0/device:GPU:1')


I0000 00:00:1782308877.133461      23 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1782308877.139300      23 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Số lượng GPU đang được đồng bộ hóa (Replicas): 2
Tổng Batch Size thực tế: 64
Tổng số món ăn Việt Nam: 30
Tổng số món ăn quốc tế (Food-101): 101
Tổng số lượng nhãn sau khi gộp: 131
Đã lưu cấu trúc nhãn vào /kaggle/working/labels.txt

--- Đang quét và gom toàn bộ file ảnh ---

--- Tìm thấy 126136 ảnh. Ép TensorFlow soi từng ảnh một... ---
(Quá trình này tốn khoảng 3-5 phút, hãy kiên nhẫn đi pha tách trà nhé!)


Corrupt JPEG data: 229 extraneous bytes before marker 0xd9
Corrupt JPEG data: 9 extraneous bytes before marker 0xe2


...

Corrupt JPEG data: 229 extraneous bytes before marker 0xd9


1577/1577 ━━━━━━━━━━━━━━━━━━━━ 838s 531ms/step - accuracy: 0.8749 - loss: 0.4214 - val_accuracy: 0.7462 - val_loss: 0.9914 - learning_rate: 4.0000e-06

--- Đang tiến hành nén và chuyển đổi sang file .tflite cho Android ---
INFO:tensorflow:Assets written to: /tmp/tmp9tcw_tg8/assets


INFO:tensorflow:Assets written to: /tmp/tmp9tcw_tg8/assets


Saved artifact at '/tmp/tmp9tcw_tg8'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name='keras_tensor_158')
Output Type:
  TensorSpec(shape=(None, 131), dtype=tf.float32, name=None)
Captures:
  140316102808912: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140313247103504: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140310142546576: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140310142543504: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140310142548304: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140309775074704: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140309775085456: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140309775088720: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140309775078352: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140309775074896: TensorSpec(shape=(), dtype=tf.resource, name=None)
  14030977

W0000 00:00:1782324347.500078      23 tf_tfl_flatbuffer_helpers.cc:365] Ignored output_format.
W0000 00:00:1782324347.500191      23 tf_tfl_flatbuffer_helpers.cc:368] Ignored drop_control_dependency.
I0000 00:00:1782324347.678785      23 mlir_graph_optimization_pass.cc:425] MLIR V1 optimization pass is not enabled
